In [ ]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
import base64
import mimetypes
import pandas as pd
import re
import json
import os

In [ ]:
"""
An Excel file is expected with the following columns:
compound, sentence, image1_name, image1_name_path, image1_caption, image2_name, image2_name_path, image2_caption, image3_name, image3_name_path, image3_caption, image4_name, image4_name_path, image4_caption, image5_name, image5_name_path, image5_caption.

The compound column contains the PIE, and the sentence column contains the context sentence in which the PIE appears. For each image x, imagex_name stores the image filename, imagex_name_path stores the path to the image, and imagex_caption provides a textual description of the image.
"""
df = pd.read_excel("resolved_image_paths.xlsx")
df

In [ ]:
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [ ]:
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"

In [ ]:
MODEL_TYPE = "GEMINI"
#model = ChatOpenAI(model="o3-2025-04-16")

model = ChatGoogleGenerativeAI(
    model="gemini-3-pro-preview",
    api_key="GEMINI_API_KEY"
)

## Step 1: Sense Predictions

In [ ]:
template = """
You are a professional linguist specializing in figurative language, with expertise in identifying idiomatic versus literal usage of potentially idiomatic expressions (PIEs).
Given a context sentence and a PIE, determine whether the PIE is used idiomatically or literally.

For each input, follow these steps:
1.Interpret the meaning contributed by the PIE within the sentence.
2.Decide whether this meaning is literal or idiomatic.
3.Return your answer in strict JSON format with the fields:

  "hasIdiom": 1 or 0, // 1 if idiomatic sense; 0 otherwise.
  "explanation": "Explain your reasoning."

Now analyze the following sentence and decide whether the PIE {compound} is used idiomatically or literally:
"{question}"
"""

prompt = ChatPromptTemplate.from_template(template)
chain = prompt | model

def extract_json_response(res):
    pattern = r'\{\s*"hasIdiom"\s*:\s*\d+\s*,\s*"explanation"\s*:\s*"([^"\\]*(\\.[^"\\]*)*)"\s*\}'
    match = re.search(pattern, res)
    result = None
    if match:
        json_str = match.group()
        result = json.loads(json_str)
    return result

In [ ]:
has_idiom = []
explanations = []

for index, row in df.iterrows():
    print("\n\n", index)
    res = chain.invoke({"compound": row["compound"], "question": row["sentence"]}).content
    if MODEL_TYPE == "GEMINI":
        res_dict = extract_json_response(res[0]["text"])
    elif MODEL_TYPE == "GPT":
        res_dict = extract_json_response(res)

    if res_dict == None:
        has_idiom.append("Wrong formatting")
        explanations.append("Wrong formatting")
        print("Wrong formatting")
        continue
    print(res_dict)
    if res_dict["hasIdiom"] == 1:
        has_idiom.append("idiomatic")
    elif res_dict["hasIdiom"] == 0:
        has_idiom.append("literal")
    explanations.append(res_dict["explanation"])

In [ ]:
df["pred"] = has_idiom
df["class_explanation"] = explanations

## Step 2: Caption Refinement

In [ ]:
def extract_json_response(res):
    pattern = re.compile(
    r'^\s*\{\s*"enhanced_caption"\s*:\s*"(?P<enhanced_caption>(?:\\.|[^"\\])*)"\s*\}\s*$',
    re.DOTALL
    )
    match = re.search(pattern, res)
    result = None
    if match:
        try:
            json_str = match.group()
            result = json.loads(json_str)
        except:
            result = None
    return result

def encode_image_to_data_url(image_path):
    mime, _ = mimetypes.guess_type(image_path)
    if mime is None:
        mime = "image/png"  # fallback

    with open(image_path, "rb") as f:
        data = base64.b64encode(f.read()).decode()

    return f"data:{mime};base64,{data}"

In [ ]:
def build_multimodal_message(img_path, caption):
    blocks = []

    blocks.append({
        "type": "text",
        "text": f"""
            You are a professional and skilled vision-language analyst.
            You will be given:
            - Image
            - Original Image Caption
            Your tasks are:
            1. Enhanced Caption
            - Expand the original caption into a more detailed description.
            - Add concrete visual details (objects, actions, setting, mood).
            - Add inferred context only when strongly supported by the image.
            - Keep the tone concise, accurate, and vivid.
            2. Produce your answer strictly in the following JSON format:
                "enhanced_caption": "Enhanced caption",
        """
    })

    ident = f"Image is"
    data_url = encode_image_to_data_url(img_path)
    blocks.append({"type": "text", "text": ident})
    blocks.append(
        {"type": "image_url", "image_url": {"url": data_url}}
    )
    blocks.append({"type": "text", "text": f"Caption is {caption}"})

    return blocks

In [ ]:
enhanced_captions = []

for index, row in df.iterrows():
    print(index)
    compound = row["compound"]

    image_paths = [ row["image1_name_path"], row["image2_name_path"], row["image3_name_path"], row["image4_name_path"], row["image5_name_path"]]

    captions = [row["image1_caption"], row["image2_caption"], row["image3_caption"], row["image4_caption"], row["image5_caption"]]

    caption = {}

    for i in range(len(captions)):
        message = HumanMessage(
            content=build_multimodal_message(image_paths[i], captions[i])
        )

        resp = model.invoke([message]).content

        if MODEL_TYPE == "GEMINI":
            res_dict = extract_json_response(resp[0]["text"])
        elif MODEL_TYPE == "GPT":
            res_dict = extract_json_response(resp)

        print(res_dict)
        cap = f"img{i}_caption"

        if res_dict == None:
            caption[cap] = "Wrong formatting"
            print("Wrong formatting")
            continue

        caption[cap] = res_dict["enhanced_caption"]

    enhanced_captions.append(caption)

In [ ]:
df_caps = pd.DataFrame(enhanced_captions)

rename_map = {
    "img0_caption": "img1_extended_caption",
    "img1_caption": "img2_extended_caption",
    "img2_caption": "img3_extended_caption",
    "img3_caption": "img4_extended_caption",
    "img4_caption": "img5_extended_caption",
}

df_caps = df_caps.rename(columns=rename_map)

df = pd.concat([df, df_caps], axis=1)
df

## Step 3: Image Classifications

In [ ]:
def extract_json_response(res):
    pattern = re.compile(
    r'\{\s*'
    r'"?explanation"?\s*:\s*"?(?P<explanation>.*?)"?\s*,\s*'
    r'"?image_ranks"?\s*:\s*\[\s*'
    r'("?(img[1-5])"?\s*,\s*){4}'
    r'"?(img[1-5])"?'
    r'\s*\]\s*'
    r'\}',
    re.DOTALL
    )
    match = re.search(pattern, res)
    result = None
    if match:
        try:
            json_str = match.group()
            result = json.loads(json_str)
        except:
            result = None
    return result

def encode_image_to_data_url(image_path):
    mime, _ = mimetypes.guess_type(image_path)
    if mime is None:
        mime = "image/png"  # fallback

    with open(image_path, "rb") as f:
        data = base64.b64encode(f.read()).decode()

    return f"data:{mime};base64,{data}"

In [ ]:
def build_multimodal_message(compound, image_paths, captions):
    """
    images: dict like {"img1": url_or_base64, ..., "img5": url_or_base64}
    captions: dict like {"caption1": "...", ..., "caption5": "..."}
    """

    blocks = []

    blocks.append({
        "type": "text",
        "text": f"""
            You are a professional linguist specializing in the interpretation of potentially idiomatic expressions (PIEs).
            You will be given:
            - A potentially idiomatic expression (PIE)
            - Five images and their captions
            (These five images span: idiomatic synonyms of the PIE, idiomatic-related visuals, literal-related visuals, literal synonyms, and a distractor.)

            Your tasks are:
            1. Analyze each of the five images and their captions.
            - Describe what each image depicts.
            - Explain how (or whether) it relates to the meaning of the PIE in context.
            2. Classify each image into exactly one of the following semantic categories:
            - Idiomatic Synonym (a visual depiction closely matching the idiomatic meaning)
            - Idiomatic-Related (conceptually related to the idiomatic meaning, but not a direct synonym)
            - Literal-Synonym (depicts the literal meaning directly or via synonyms)
            - Literal-Related (related to the literal meaning, but not a direct synonym)
            - Distractor (unrelated to either literal or idiomatic meaning)
            3. Rank the images in the following sequence:
            - 1) Idiomatic Synonym → 2) Idiomatic-Related → 3) Literal-Related → 4) Literal-Synonym → 5) Distractor
            4. Produce your answer strictly in the following JSON format:
                "explanation": "Concise explanation of your reasoning.",
                "image_ranks": ["imgX", "imgY", "imgZ", "imgW", "imgV"]
            image_ranks: Example: ["img2", "img3", "img4", "img1", "img5"]

            PIE is {compound}.
        """
    })

    for i in range(1, 6):
        ident = f"img{i}"
        cap = f"caption{i}"
        data_url = encode_image_to_data_url(image_paths[ident])
        blocks.append({"type": "text", "text": ident})
        blocks.append(
            {"type": "image_url", "image_url": {"url": data_url}}
        )
        blocks.append({"type": "text", "text": captions[cap]})

    return blocks

In [ ]:
explanations = []
image_ranks = []


for index, row in df.iterrows():
    print(index)
    compound = row["compound"]

    image_paths = {
        "img1": row["image1_name_path"],
        "img2": row["image2_name_path"],
        "img3": row["image3_name_path"],
        "img4": row["image4_name_path"],
        "img5": row["image5_name_path"],
    }

    captions = {
        "caption1": row["img1_extended_caption"],
        "caption2": row["img2_extended_caption"],
        "caption3": row["img3_extended_caption"],
        "caption4": row["img4_extended_caption"],
        "caption5": row["img5_extended_caption"],
    }

    message = HumanMessage(
        content=build_multimodal_message(compound, image_paths, captions)
    )

    resp = model.invoke([message]).content
    if MODEL_TYPE == "GEMINI":
        res_dict = extract_json_response(resp[0]["text"])
    elif MODEL_TYPE == "GPT":
        res_dict = extract_json_response(resp)

    if res_dict == None:
        explanations.append("Wrong formatting")
        image_ranks.append("Wrong formatting")
        print("Wrong formatting")
        continue
    explanations.append(res_dict["explanation"])
    image_ranks.append(res_dict["image_ranks"])
    print(resp)

In [ ]:
df["image_classification_explanations"] = explanations
df["image_classifications"] = image_ranks

In [ ]:
category_order = [
    "idiomatic_synonym",
    "idiomatic_related",
    "literal_related",
    "literal_synonym",
    "distractor"
]

def extract_classes(order_list):
    # Map imageID → class based on position
    mapping = {order_list[i]: category_order[i] for i in range(5)}

    # Generate column values in img1..img5 order
    return pd.Series([mapping[f"img{i}"] for i in range(1, 6)])

df[["img1_class", "img2_class", "img3_class", "img4_class", "img5_class"]] = \
    df["image_classifications"].apply(extract_classes)

## Step 4: Image Ranking

In [ ]:
def build_multimodal_message(compound, context_sentence, s_type, s_type_exp, img_class_exp, image_paths, captions, image_classes):
    """
    images: dict like {"img1": url_or_base64, ..., "img5": url_or_base64}
    captions: dict like {"caption1": "...", ..., "caption5": "..."}
    """

    blocks = []

    blocks.append({
        "type": "text",
        "text": f"""
            You are a professional linguist specializing in the interpretation of potentially idiomatic expressions (PIEs).
            You will be given the following information:
            - PIE: the potentially idiomatic expression.
            - Context Sentence: a sentence containing the PIE.
            - Usage Type: a label indicating whether the PIE is used idiomatically or literally in the context sentence.
            - Explanation of Usage: an explanation of why the usage type was assigned.
            - Five Images that span idiomatic synonyms, idiomatic-related visuals, literal synonyms, literal-related visuals, and one distractor.
                Each image includes:
                - Image ID
                - Image Caption
                - Image Class (one of: idiomatic-synonym, idiomatic-related, literal-synonym, literal-related, distractor)
            - Explanation of the Image Classes

            Your tasks:
            1. Interpret the PIE in context.
            - Explain the intended meaning of the PIE in the provided sentence, considering its usage type (literal or idiomatic).
            2. Identify the most relevant visual representations.
            - Describe what kinds of images would best depict the intended meaning, given the usage type.
            3. Evaluate each of the five images.
                For each image:
                    - Describe its visual content.
                    - Explain how well it matches the intended meaning of the PIE in this context, taking into account its class and the class explanation.
            4. Rank the five images from best to worst.
                - Order the images (img1–img5) by how well they convey the intended meaning in the context sentence, from most relevant (rank 1) to least relevant (rank 5).
                - Provide a one-sentence justification for each ranking position.
            5. Produce your answer strictly in the following JSON format:
                "explanation": "Concise explanation of your reasoning.",
                "image_ranks": ["imgX", "imgY", "imgZ", "imgW", "imgV"]
            image_ranks: Example: ["img2", "img3", "img4", "img1", "img5"]

            You are given the following information:

            PIE: {compound}.
            Context Sentence: {context_sentence}
            Usage Type: {s_type}
            Explanation of Usage: {s_type_exp}
        """
    })

    for i in range(1, 6):
        ident = f"img{i}"
        cap = f"caption{i}"
        img_class = f"img{i}_class"
        data_url = encode_image_to_data_url(image_paths[ident])
        blocks.append({"type": "text", "text": ident})
        blocks.append(
            {"type": "image_url", "image_url": {"url": data_url}}
        )
        blocks.append({"type": "text", "text": captions[cap]})
        blocks.append({"type": "text", "text": image_classes[img_class]})

    blocks.append({"type": "text", "text":
                  f"""
                  Explanation of Image Classes: {img_class_exp}
                  """})

    return blocks

In [ ]:
explanations = []
image_ranks = []


for index, row in df.iterrows():
    print(index)
    compound = row["compound"]
    context_sentence = row["sentence"]
    s_type = row["pred"]
    s_type_exp = row["class_explanation"]
    img_class_exp = row["image_classification_explanations"]


    image_paths = {
        "img1": row["image1_name_path"],
        "img2": row["image2_name_path"],
        "img3": row["image3_name_path"],
        "img4": row["image4_name_path"],
        "img5": row["image5_name_path"],
    }

    extended_captions = {
        "caption1": row["img1_extended_caption"],
        "caption2": row["img2_extended_caption"],
        "caption3": row["img3_extended_caption"],
        "caption4": row["img4_extended_caption"],
        "caption5": row["img5_extended_caption"],
    }

    image_classes = {
        "img1_class": row["img1_class"],
        "img2_class": row["img2_class"],
        "img3_class": row["img3_class"],
        "img4_class": row["img4_class"],
        "img5_class": row["img5_class"]
    }

    message = HumanMessage(
        content=build_multimodal_message(compound, context_sentence, s_type, s_type_exp, img_class_exp, image_paths, extended_captions, image_classes)
    )

    resp = model.invoke([message]).content
    res_dict = extract_json_response(resp)

    if res_dict == None:
        explanations.append("Wrong formatting")
        image_ranks.append("Wrong formatting")
        print("Wrong formatting")
        continue
    explanations.append(res_dict["explanation"])
    image_ranks.append(res_dict["image_ranks"])
    print(resp)

In [ ]:
df["image_ranks"] = image_ranks
df["rank_explanation"] = explanations

In [ ]:
df.to_excel("results_4step.xlsx")